# The Alarm Is Not the Change

A reproducible notebook for [the long-form article](article.md). An online alarm is an observation-dependent stopping time, not a measurement of the true change time. We generate entirely **synthetic** Gaussian data and compare four sequential statistics at matched **200-event** false-alarm risk. This notebook uses the quick simulation; the publication tables and eight figures come from `python analysis.py`.

## What is calibrated?

Every candidate threshold is chosen as the 90th percentile of the **largest statistic on each** of 1,000 independent null streams of length 200. Thus we target P(any false alarm before observation 200) = 0.10 under iid N(0,1) data. This is not a matched average run length (ARL0). A separate iid validation set checks the chosen thresholds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from analysis import METHODS, LABELS, CHANGE_INDEX, run_study, detector_statistics

study = run_study(quick=True)
for row in study.calibration_rows:
    print(row['method'], 'threshold:', round(float(row['threshold']), 3), 'iid false alarm by 200:', round(float(row['validated_iid_false_alarm_probability']), 3))
assert len(study.calibration_rows) == 4

## The alarm is not the time of the change

We know that the synthetic mean changes at event index 80. Each statistic is causal: at time t it can only use x[0], ..., x[t]. Its alarm may occur later, earlier through noise, or never. This single path is illustrative, not a performance estimate.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)
for ax, method in zip(axes.flat, METHODS):
    ax.plot(study.example_statistics[method])
    ax.axhline(study.thresholds[method], linestyle='--', color='tab:red', label='Horizon-calibrated limit')
    ax.axvline(CHANGE_INDEX, linestyle=':', color='black', label='True change')
    ax.set_title(LABELS[method])
    ax.set_xlabel('Event index')
    ax.grid(alpha=.2)
axes[0, 0].legend(fontsize=8)
plt.show()

## Conditional detection and right-censoring

The detection endpoint below is the fraction of **streams that have not alarmed before the known change** whose first alarm falls within the following 50 observations. No first alarm in that interval is recorded as not detected. The median delay among detections describes only the detected subset, not all streams. Do not turn it into an unconditional operational service-level promise.

In [ ]:
for scenario in ('mean_0_5', 'mean_1_0', 'single_spike', 'variance_1_7'):
    print('\n', scenario)
    for row in study.scenario_rows:
        if row['scenario'] == scenario:
            print(' ', row['method'], 'within 50:', round(float(row['conditional_detection_50']), 3), 'within 5:', round(float(row['conditional_detection_5']), 3))
assert all(0 <= float(r['conditional_detection_50']) <= 1 for r in study.scenario_rows)

## A hidden production risk: temporal dependence

The iid Gaussian null and the stationary Gaussian AR(1) null have the **same N(0,1) marginal distribution**. The second has positive serial correlation phi = 0.6. We deliberately reuse the iid-calibrated thresholds for both. This isolates dependence as a failure mode; in a real system we would have to recalibrate using an appropriate dependence model or blocks.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
positions = np.arange(len(METHODS))
for process, title, offset in [('iid_gaussian', 'iid N(0,1)', -.18), ('ar1_gaussian_phi_0_6', 'AR(1), phi=0.6', .18)]:
    rates = [float(next(r['false_alarm_probability_by_200'] for r in study.dependence_rows if r['method'] == method and r['null_process'] == process)) for method in METHODS]
    ax.bar(positions + offset, rates, .36, label=title)
ax.axhline(.10, color='black', linestyle='--', label='iid calibration target')
ax.set_xticks(positions, [LABELS[m] for m in METHODS], rotation=15)
ax.set_ylim(0, 1)
ax.set_ylabel('Probability of a false alarm by 200 observations')
ax.legend()
ax.grid(axis='y', alpha=.2)
plt.show()

## Reproduction and limits

Run `python analysis.py` for the full seeded results and eight publication figures. Run `python analysis.py --quick` for disposable, suffix-labelled smoke output. This example assumes known mean/variance, fixed iid calibration horizon, one-sided thresholds and no missingness. It is a method comparison, not evidence of performance on any real operational stream. More information and original research sources are in [the article](article.md).